In [11]:
import dgl
import torch
import random


graph_path = "/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/kb_acyclic_reg_cxg.dgl"
g = dgl.load_graphs(graph_path)[0][0]
target_etype = 'regulate'
srctype, etype, dsttype = g.to_canonical_etype(target_etype)
src, dst = g.edges(etype=target_etype)
num_edges = len(src)
num_shuffle = int(num_edges * 0.3)
edge_idx = list(range(num_edges))
shuffle_idx = random.sample(edge_idx, num_shuffle)

src_shuffle = src[shuffle_idx]
dst_shuffle = dst[shuffle_idx]

num_dst_nodes = g.num_nodes(dsttype)
new_dst = torch.randint(0, num_dst_nodes, (num_shuffle,))
new_dst = torch.where(new_dst != src_shuffle, new_dst, (new_dst + 1) % num_dst_nodes)

keep_idx = list(set(edge_idx) - set(shuffle_idx))
final_src = torch.cat([src[keep_idx], src_shuffle])
final_dst = torch.cat([dst[keep_idx], new_dst])

data_dict = {
    et: g.edges(etype=et) for et in g.etypes if et != target_etype
}

data_dict[(srctype, etype, dsttype)] = (final_src, final_dst)

new_g = dgl.heterograph(data_dict, num_nodes_dict={nt: g.num_nodes(nt) for nt in g.ntypes})

for ntype in g.ntypes:
    for key, val in g.nodes[ntype].data.items():
        new_g.nodes[ntype].data[key] = val

for et in g.etypes:
    if et == target_etype:
        for key, val in g.edges[et].data.items():
            keep_feat = val[keep_idx]
            pad_feat = torch.zeros((num_shuffle,) + keep_feat.shape[1:], dtype=val.dtype)
            new_g.edges[et].data[key] = torch.cat([keep_feat, pad_feat])
    else:
        for key, val in g.edges[et].data.items():
            new_g.edges[et].data[key] = val

save_path = "/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/shuffled_0.3_kb_acyclic_reg_cxg.dgl"
dgl.save_graphs(save_path, [new_g])
print(f"Shuffled graph saved to: {save_path}")


Shuffled graph saved to: /home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/shuffled_0.3_kb_acyclic_reg_cxg.dgl


In [12]:
import dgl
import networkx as nx

def remove_cycle(graph):
    nx_g = dgl.to_networkx(graph)
    while True:
        try:
            cycle = nx.find_cycle(nx_g, orientation='original')
            print(f'Detect cycle: {cycle}')
            for u, v, _, _ in cycle:
                # Remove edge from the DGL graph
                graph.remove_edges(graph.edge_ids(u, v))
            nx_g = dgl.to_networkx(graph)
        except:
            print('no cycle detected')
            break
    return graph

In [13]:
graph=dgl.load_graphs("/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/shuffled_0.3_kb_acyclic_reg_cxg.dgl")[0][0]
regulate_subg = dgl.edge_type_subgraph(graph, etypes=['regulate'])
acyclic_regulate_graph=remove_cycle(regulate_subg)
dgl.save_graphs('/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/shuffled_0.3_kb_acyclic_reg_cxg.dgl',acyclic_regulate_graph)

Detect cycle: [(60694, 9583, 0, 'forward'), (9583, 60694, 0, 'forward')]
Detect cycle: [(60694, 3263, 0, 'forward'), (3263, 60694, 0, 'forward')]
Detect cycle: [(36233, 35943, 0, 'forward'), (35943, 32066, 0, 'forward'), (32066, 2758, 0, 'forward'), (2758, 35711, 0, 'forward'), (35711, 35831, 0, 'forward'), (35831, 8123, 0, 'forward'), (8123, 10565, 0, 'forward'), (10565, 36233, 0, 'forward')]
Detect cycle: [(9567, 36112, 0, 'forward'), (36112, 36301, 0, 'forward'), (36301, 33208, 0, 'forward'), (33208, 2758, 0, 'forward'), (2758, 3025, 0, 'forward'), (3025, 9335, 0, 'forward'), (9335, 36343, 0, 'forward'), (36343, 8496, 0, 'forward'), (8496, 36028, 0, 'forward'), (36028, 6294, 0, 'forward'), (6294, 36410, 0, 'forward'), (36410, 36307, 0, 'forward'), (36307, 36184, 0, 'forward'), (36184, 11194, 0, 'forward'), (11194, 8176, 0, 'forward'), (8176, 33104, 0, 'forward'), (33104, 31242, 0, 'forward'), (31242, 32206, 0, 'forward'), (32206, 36102, 0, 'forward'), (36102, 9567, 0, 'forward')]
De

In [14]:
import dgl
import torch
import random


graph_path = "/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/kb_acyclic_reg_cxg.dgl"
g = dgl.load_graphs(graph_path)[0][0]


target_etype = 'regulate'

srctype, etype, dsttype = g.to_canonical_etype(target_etype)


src, dst = g.edges(etype=target_etype)
num_edges = len(src)
num_remove = int(num_edges * 0.3)
edge_idx = list(range(num_edges))
remove_idx = random.sample(edge_idx, num_remove)


keep_idx = list(set(edge_idx) - set(remove_idx))

final_src = src[keep_idx]
final_dst = dst[keep_idx]


data_dict = {
    et: g.edges(etype=et) for et in g.etypes if et != target_etype
}
data_dict[(srctype, etype, dsttype)] = (final_src, final_dst)

new_g = dgl.heterograph(data_dict, num_nodes_dict={nt: g.num_nodes(nt) for nt in g.ntypes})


for ntype in g.ntypes:
    for key, val in g.nodes[ntype].data.items():
        new_g.nodes[ntype].data[key] = val

for et in g.etypes:
    if et == target_etype:
        for key, val in g.edges[et].data.items():
            new_g.edges[et].data[key] = val[keep_idx]
    else:
        for key, val in g.edges[et].data.items():
            new_g.edges[et].data[key] = val


save_path = "/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/removed_0.3_kb_acyclic_reg_cxg.dgl"
dgl.save_graphs(save_path, [new_g])
print(f"Graph with 30% edges removed saved to: {save_path}")


Graph with 30% edges removed saved to: /home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/removed_0.3_kb_acyclic_reg_cxg.dgl


In [15]:
graph=dgl.load_graphs("/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/removed_0.3_kb_acyclic_reg_cxg.dgl")[0][0]
regulate_subg = dgl.edge_type_subgraph(graph, etypes=['regulate'])
acyclic_regulate_graph=remove_cycle(regulate_subg)
dgl.save_graphs('/home/share/huadjyin/home/s_qiuping1/workspace/RegFormer/example/data/graph/removed_0.3_kb_acyclic_reg_cxg.dgl',acyclic_regulate_graph)

no cycle detected
